In [ ]:
import argparse
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

In [ ]:
# -*- coding: utf-8 -*-
"""
Autoencoder Training Pipeline
Dedicated script for compiling, training, and exporting the image autoencoder weights.
"""



# Configuration
BLOCK_SIZE = 256
LATENT_DIM = 128

def parse_args():
    parser = argparse.ArgumentParser(description="Train the Image Autoencoder.")
    parser.add_argument("--data-dir", type=str, default="images/", help="Directory containing training images")
    parser.add_argument("--epochs", type=int, default=100, help="Number of epochs to train the model")
    parser.add_argument("--batch-size", type=int, default=32, help="Training batch size")
    return parser.parse_args()

@tf.keras.utils.register_keras_serializable()
def custom_image_loss(y_true, y_pred):
    """Compound Loss: Combines Pixel Accuracy (MSE) with Perceptual Quality (SSIM)."""
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    ssim = tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    return 0.8 * mse + 0.2 * (1.0 - ssim)

def load_and_split_dataset(data_dir, batch_size=8):
    """Loads images from a directory, processes them, and splits into train/val/test."""
    print(f"[*] Scanning for training data in '{data_dir}'...")

    if not os.path.exists(data_dir):
        print(f"[!] Warning: Directory '{data_dir}' not found. Generating synthetic dummy dataset for testing.")
        dummy_x = np.random.uniform(0.0, 1.0, (100, BLOCK_SIZE, BLOCK_SIZE, 3)).astype(np.float32)
        dataset = tf.data.Dataset.from_tensor_slices((dummy_x, dummy_x)).batch(batch_size)
        return dataset, dataset, dataset

    # In a real environment, you would use:
    # dataset = tf.keras.utils.image_dataset_from_directory(data_dir, image_size=(BLOCK_SIZE, BLOCK_SIZE), batch_size=batch_size)
    # return train_ds, val_ds, test_ds

    dataset = tf.data.Dataset.from_tensor_slices(([], [])).batch(batch_size)
    return dataset, dataset, dataset

In [ ]:
#Encoder Section
def build_autoencoder():
    """Builds a functioning Autoencoder using the MobileNetV2 backbone."""
    print(f"[*] Building Autoencoder with MobileNetV2 backbone for ({BLOCK_SIZE}x{BLOCK_SIZE}) blocks...")

    base_model = MobileNetV2(input_shape=(BLOCK_SIZE, BLOCK_SIZE, 3), include_top=False, weights='imagenet')
    base_model.trainable = False  # Freeze backbone for stable training

    inputs = layers.Input(shape=(BLOCK_SIZE, BLOCK_SIZE, 3))
    x_scaled = layers.Lambda(lambda x: x * 2.0 - 1.0)(inputs)
    x = base_model(x_scaled, training=False)
    x = layers.Flatten()(x)
    latent = layers.Dense(LATENT_DIM, name="bottleneck")(x)
    encoder = Model(inputs, latent, name="Encoder")

    latent_inputs = layers.Input(shape=(LATENT_DIM,))
    x = layers.Dense(8 * 8 * 128, activation='relu')(latent_inputs)
    x = layers.Reshape((8, 8, 128))(x)

    for filters in [64, 32, 16, 8, 8]:
        x = layers.Conv2DTranspose(filters, kernel_size=3, strides=2, padding='same', activation='relu')(x)

    outputs = layers.Conv2D(3, kernel_size=3, padding='same', activation='sigmoid')(x)
    decoder = Model(latent_inputs, outputs, name="Decoder")

    autoencoder = Model(inputs, decoder(encoder(inputs)), name="Autoencoder")
    return autoencoder

def main():
    args = parse_args()
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"[*] High-end hardware detected: {len(gpus)} GPU(s) available. Enabling memory growth.")
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

    print("\n=== INITIATING TRAINING PIPELINE ===")

    # 1. Load and Split Data
    train_data, val_data, _ = load_and_split_dataset(args.data_dir, batch_size=args.batch_size)

    # 2. Build and Compile Model
    autoencoder = build_autoencoder()
    print("[*] Compiling model with Adam optimizer and custom MSE+SSIM loss...")
    autoencoder.compile(optimizer='adam', loss=custom_image_loss)

    # 3. Train
    print(f"[*] Commencing training for {args.epochs} epochs...")
    autoencoder.fit(train_data, validation_data=val_data, epochs=args.epochs)

    # 4. Save Weights
    weight_path = "production_model.h5"
    autoencoder.save_weights(weight_path)
    print(f"\n[+] SUCCESS: Training complete. Model weights saved to '{weight_path}'.")
    print("[+] You can now run the 'dynamic_grid_compressor.py' script for inference!")

if __name__ == "__main__":
    main()